In [1]:
import pandas as pd
import numpy as np
import re
import string

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\admin\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\admin\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\admin\AppData\Roaming\nltk_data...


True

In [3]:
df = pd.read_csv("../dataset/news_dataset.csv")

print("Dataset loaded successfully!")
print("Shape:", df.shape)

Dataset loaded successfully!
Shape: (44898, 5)


In [4]:
df.head()

,title,text,subject,date,label
0,It’s Really Happening: Trump Adviser Lays Out...,"Well, that didn t take long. In the short time...",News,"November 16, 2016",0
1,Republican attempt to deflect Trump-Russia pro...,(Reuters) - Republican lawmaker Devin Nunes’ i...,politicsNews,"September 11, 2017",1
2,Trump says churches should get FEMA funds for ...,WASHINGTON (Reuters) - President Donald Trump ...,politicsNews,"September 9, 2017",1
3,Trump Loves To Say The New York Times Is ‘Fai...,Print journalism and longstanding papers have ...,News,"August 7, 2017",0
4,House Speaker Ryan briefed Trump on healthcare...,WASHINGTON (Reuters) - President Donald Trump ...,politicsNews,"March 24, 2017",1


In [5]:
print("Columns:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

Columns:
['title', 'text', 'subject', 'date', 'label']

Missing values:
title      0
text       0
subject    0
date       0
label      0
dtype: int64

Duplicate rows:
209


In [6]:
print("Rows before removing duplicates:", len(df))

df = df.drop_duplicates().reset_index(drop=True)

print("Rows after removing duplicates:", len(df))

Rows before removing duplicates: 44898
Rows after removing duplicates: 44689


In [8]:
df.isnull().sum()

title      0
text       0
subject    0
date       0
label      0
dtype: int64

In [11]:
df["title"] = df["title"].fillna("")
df["text"] = df["text"].fillna("")

df.isnull().sum()

title      0
text       0
subject    0
date       0
label      0
dtype: int64

In [12]:
df["content"] = df["title"] + " " + df["text"]

In [13]:
df[["title", "text", "content"]].head()

,title,text,content
0,It’s Really Happening: Trump Adviser Lays Out...,"Well, that didn t take long. In the short time...",It’s Really Happening: Trump Adviser Lays Out...
1,Republican attempt to deflect Trump-Russia pro...,(Reuters) - Republican lawmaker Devin Nunes’ i...,Republican attempt to deflect Trump-Russia pro...
2,Trump says churches should get FEMA funds for ...,WASHINGTON (Reuters) - President Donald Trump ...,Trump says churches should get FEMA funds for ...
3,Trump Loves To Say The New York Times Is ‘Fai...,Print journalism and longstanding papers have ...,Trump Loves To Say The New York Times Is ‘Fai...
4,House Speaker Ryan briefed Trump on healthcare...,WASHINGTON (Reuters) - President Donald Trump ...,House Speaker Ryan briefed Trump on healthcare...


In [14]:
stop_words = set(stopwords.words("english"))

lemmatizer = WordNetLemmatizer()

print("Number of English stopwords:", len(stop_words))

Number of English stopwords: 198


In [15]:
def clean_text(text):
    
    # Convert to string
    text = str(text)
    
    # Convert to lowercase
    text = text.lower()
    
    # Remove URLs
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)
    
    # Remove HTML tags
    text = re.sub(r"<.*?>", "", text)
    
    # Remove punctuation
    text = text.translate(
        str.maketrans("", "", string.punctuation)
    )
    
    # Remove numbers
    text = re.sub(r"\d+", "", text)
    
    # Remove extra whitespace
    text = re.sub(r"\s+", " ", text).strip()
    
    # Tokenization
    words = text.split()
    
    # Remove stopwords and lemmatize
    words = [
        lemmatizer.lemmatize(word)
        for word in words
        if word not in stop_words
    ]
    
    # Join words back together
    return " ".join(words)

In [16]:
sample_text = """
BREAKING NEWS!!! The Government has announced a NEW policy.
Visit https://example.com for more information.
"""

print("Original:")
print(sample_text)

print("\nCleaned:")
print(clean_text(sample_text))

Original:

BREAKING NEWS!!! The Government has announced a NEW policy.
Visit https://example.com for more information.


Cleaned:
breaking news government announced new policy visit information


In [17]:
df["clean_content"] = df["content"].apply(clean_text)

In [18]:
print("ORIGINAL TEXT:")
print(df["content"].iloc[0])

print("\n" + "=" * 80 + "\n")

print("CLEANED TEXT:")
print(df["clean_content"].iloc[0])

ORIGINAL TEXT:
 It’s Really Happening: Trump Adviser Lays Out Plan For National Muslim Registry Well, that didn t take long. In the short time since Americans kinda-sorta elected Donald Trump to be Pussygrabber-in-Chief, Trump has appointed a bona fide white nationalist to a high-level position and perform numerous other actions that months ago were considered extremely unlikely nightmare scenarios   and the latest seems to be that the Muslim registry the media told us Trump was not serious about might actually happen.Under the Trump regime, Muslim immigrants coming to the United States from what Trump and Pals call  terror-prone  countries will soon have to register and check in regularly with the government if Trump adviser and Kansas Secretary of State, Kris Kobach, has his way.Kobach says that the Trump administration aims to quickly waste taxpayer dollars on The Wall and that they are moving forward on a proposal to get things rolling with a registry for immigrants from Muslim cou

In [19]:
df["clean_length"] = df["clean_content"].str.len()

df["clean_length"].describe()

count    44689.000000
mean      1770.336145
std       1515.903461
min          0.000000
25%        916.000000
50%       1555.000000
75%       2196.000000
max      37930.000000
Name: clean_length, dtype: float64

In [20]:
empty_articles = df[
    df["clean_content"].str.strip() == ""
]

print("Empty articles:", len(empty_articles))

Empty articles: 9


In [21]:
df = df[
    df["clean_content"].str.strip() != ""
].reset_index(drop=True)

print("Dataset shape after removing empty articles:", df.shape)

Dataset shape after removing empty articles: (44680, 8)


In [22]:
print(df["label"].value_counts())

label
0    23469
1    21211
Name: count, dtype: int64


In [24]:
print(df["label"].value_counts(normalize = True) * 100)

label
0    52.526858
1    47.473142
Name: proportion, dtype: float64


In [25]:
df[
    ["title", "clean_content", "label"]
].head()

,title,clean_content,label
0,It’s Really Happening: Trump Adviser Lays Out...,it’s really happening trump adviser lay plan n...,0
1,Republican attempt to deflect Trump-Russia pro...,republican attempt deflect trumprussia probe c...,1
2,Trump says churches should get FEMA funds for ...,trump say church get fema fund hurricane relie...,1
3,Trump Loves To Say The New York Times Is ‘Fai...,trump love say new york time ‘failing’ – effor...,0
4,House Speaker Ryan briefed Trump on healthcare...,house speaker ryan briefed trump healthcare bi...,1


In [29]:
df = df.drop(
    columns=["subject", "date", "content", "clean_length"]
)

KeyError: "['subject', 'date', 'content', 'clean_length'] not found in axis"

In [30]:
df.columns

Index(['title', 'text', 'label', 'clean_content'], dtype='str')

In [31]:
df = df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

In [32]:
df.to_csv(
    "../dataset/cleaned_news_dataset.csv",
    index=False
)

print("Cleaned dataset saved successfully!")

Cleaned dataset saved successfully!


In [33]:
print("Final Dataset Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nMissing Values:")
print(df.isnull().sum())

print("\nLabels:")
print(df["label"].value_counts())

print("\nSample:")
display(df.head())

Final Dataset Shape: (44680, 4)

Columns:
['title', 'text', 'label', 'clean_content']

Missing Values:
title            0
text             0
label            0
clean_content    0
dtype: int64

Labels:
label
0    23469
1    21211
Name: count, dtype: int64

Sample:


,title,text,label,clean_content
0,DIRTY POOL! FBI AND DOJ Just Affirmed They Spi...,How dirty is this? The powers-that-be at the i...,0,dirty pool fbi doj affirmed spied trump withou...
1,Virginia GOP sues governor over restoring voti...,RICHMOND (Reuters) - Virginia Republicans on M...,1,virginia gop sue governor restoring voting rig...
2,OBAMA’S EMBARRASSING Farewell Interview: Mom W...,We all know the rules are a little different f...,0,obama’s embarrassing farewell interview mom “h...
3,BREAKING: WISCONSIN RECOUNT ENDS…Trump Picks U...,Green Party candidate Jill Stein successfully ...,0,breaking wisconsin recount ends…trump pick vot...
4,THE ROBBING OF INNOCENCE: 12 Yr Old Students G...,The left believes these are all perfectly acce...,0,robbing innocence yr old student given cdc sur...
